# 통신데이터 EDA

역할: T4~T27 전체 통신 데이터 구조를 훑고, EDA에서 변수 후보로 연결한 테이블의 분포와 예측용 변수 해석을 정리한다.

파일 생성은 `통신데이터_통합.ipynb`, 재사용 가능성 검증은 `통신데이터_검증.ipynb`에서 수행한다.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DATA_DIR = Path('../data')
con = duckdb.connect()

# 전체 t 데이터 EDA는 대용량 스캔이므로 기본은 False로 둔다.
# 전체 월별/지역별/목적별 집계를 다시 실행할 때만 True로 바꾼다.
RUN_FULL_T_EDA = False
RUN_FULL_T_HEAVY_EDA = False

In [ ]:
T_DATA_FILES = {
    'T4':  {'file': 't4_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '시군구', 'meaning': '도착지 + 목적', 'decision': '구조 파악 / 보조'},
    'T5':  {'file': 't5_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '행정동', 'meaning': '도착지 + 목적', 'decision': '구조 파악 / 보조'},
    'T6':  {'file': 't6_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '시군구', 'meaning': '도착지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T7':  {'file': 't7_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '행정동', 'meaning': '도착지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T8':  {'file': 't8_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '시군구', 'meaning': '출발지 + 목적', 'decision': '구조 파악 / 보조'},
    'T9':  {'file': 't9_2023_2025_all_date_final.parquet',  'format': 'parquet', 'unit': '행정동', 'meaning': '출발지 + 목적', 'decision': '구조 파악 / 보조'},
    'T10': {'file': 't10_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '시군구', 'meaning': '출발지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T11': {'file': 't11_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '행정동', 'meaning': '출발지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T12': {'file': 't12_2023_2025_all_final_v2.parquet',   'format': 'parquet', 'unit': '시군구 OD', 'meaning': '출발지->도착지 + 목적', 'decision': '구조 파악 / 보조'},
    'T13': {'file': 't13_seongnam_final.parquet',           'format': 'parquet', 'unit': '행정동 OD', 'meaning': '출발지->도착지 + 목적', 'decision': '예측 변수 후보'},
    'T14': {'file': 't14_2023_2025_all_final_v2.parquet',   'format': 'parquet', 'unit': '시군구 OD', 'meaning': '출발지->도착지 + 교통수단', 'decision': '구조 파악 / 보조'},
    'T16': {'file': 't16_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '시군구', 'meaning': '도착지 + 목적 + 체류시간', 'decision': '구조 파악 / 보조'},
    'T20': {'file': 't20_2023_2025_all_date_final.csv',     'format': 'csv', 'unit': '기타', 'meaning': '통합 CSV 참고 파일', 'decision': '참고용 / 분석 후보 제외'},
    'T21': {'file': 't21_2023_2025_all_date_final.csv',     'format': 'csv', 'unit': '기타', 'meaning': '통합 CSV 참고 파일', 'decision': '참고용 / 분석 후보 제외'},
    'T22': {'file': 't22_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '행정동', 'meaning': '시간대 + 성별/연령 + 내외국인', 'decision': '구조 파악 / 보조'},
    'T23': {'file': 't23_2023_2025_all_date_final.parquet', 'format': 'parquet', 'unit': '시군구', 'meaning': '시간대 + 목적 유동인구', 'decision': '구조 파악 / 보조'},
    'T24': {'file': 't24_seongnam_final.parquet',           'format': 'parquet', 'unit': '행정동', 'meaning': '시간대 + 목적 유동인구', 'decision': '예측 변수 후보'},
    'T25': {'file': 't25_seongnam_final.parquet',           'format': 'parquet', 'unit': '시군구 OD', 'meaning': '출발지->도착지 + 시간대 + 목적 + 교통수단', 'decision': '예측 변수 후보'},
    'T26': {'file': 't26_seongnam_final.parquet',           'format': 'parquet', 'unit': '행정동', 'meaning': '도착지 + 시간대 + 목적 + 교통수단 + 체류시간', 'decision': '예측 변수 후보'},
    'T27': {'file': 't27_seongnam_final.parquet',           'format': 'parquet', 'unit': '행정동', 'meaning': '출발지 + 시간대 + 목적 + 교통수단 + 체류시간', 'decision': '예측 변수 후보'},
}

FINAL_FILES = {k: DATA_DIR / v['file'] for k, v in T_DATA_FILES.items() if v['decision'] == '예측 변수 후보'}
all_t_inventory = pd.DataFrame([{'table': table, **info} for table, info in T_DATA_FILES.items()])
all_t_inventory

## 전체 t 데이터 구조 요약

| 데이터 | 단위 | 기준 | 주요 의미 |
|---|---|---|---|
| T4 | 시군구 | 도착지 + 목적 | 어느 구로, 어떤 목적의 사람이 유입됐는지 |
| T5 | 행정동 | 도착지 + 목적 | 어느 행정동으로, 어떤 목적의 사람이 유입됐는지 |
| T6 | 시군구 | 도착지 + 교통수단 | 어느 구로, 어떤 교통수단으로 도착했는지 |
| T7 | 행정동 | 도착지 + 교통수단 | 어느 행정동으로, 어떤 교통수단으로 도착했는지 |
| T8 | 시군구 | 출발지 + 목적 | 어느 구에서, 어떤 목적의 이동이 시작됐는지 |
| T9 | 행정동 | 출발지 + 목적 | 어느 행정동에서, 어떤 목적의 이동이 시작됐는지 |
| T10 | 시군구 | 출발지 + 교통수단 | 어느 구에서, 어떤 교통수단 이동이 시작됐는지 |
| T11 | 행정동 | 출발지 + 교통수단 | 어느 행정동에서, 어떤 교통수단 이동이 시작됐는지 |
| T12 | 시군구 OD | 출발지->도착지 + 목적 | 구 간 이동을 목적별로 본 데이터 |
| T13 | 행정동 OD | 출발지->도착지 + 목적 | 행정동 간 이동을 목적별로 본 데이터 |
| T14 | 시군구 OD | 출발지->도착지 + 교통수단 | 구 간 이동을 교통수단별로 본 데이터 |
| T16 | 시군구 | 도착지 + 목적 + 체류시간 | 어느 구에 도착해 얼마나 머무는지 |
| T20 | 기타 | 통합 CSV 참고 파일 | 참고용 통신 데이터 |
| T21 | 기타 | 통합 CSV 참고 파일 | 참고용 통신 데이터 |
| T22 | 행정동 | 시간대 + 성별/연령 + 내외국인 | 행정동별 시간대 생활/체류 인구 성격 |
| T23 | 시군구 | 시간대 + 목적 | 구 단위 시간대별 목적 유동인구 |
| T24 | 행정동 | 시간대 + 목적 | 행정동 단위 시간대별 목적 유동인구 |
| T25 | 시군구 OD | 출발지->도착지 + 시간대 + 목적 + 교통수단 | 구 간 이동을 시간·목적·교통수단까지 세분화 |
| T26 | 행정동 | 도착지 + 시간대 + 목적 + 교통수단 + 체류시간 | 행정동별 도착/체류 특성 |
| T27 | 행정동 | 출발지 + 시간대 + 목적 + 교통수단 + 체류시간 | 행정동별 출발/이탈 특성 |

In [ ]:
def p(path):
    return str(path).replace('\\', '/')


def scan_sql(table):
    info = T_DATA_FILES[table]
    path = p(DATA_DIR / info['file'])
    if info['format'] == 'csv':
        return f"read_csv_auto('{path}', header=true)"
    return f"read_parquet('{path}')"


def table_columns(table):
    return con.execute(f"DESCRIBE SELECT * FROM {scan_sql(table)}").fetchdf()['column_name'].tolist()


def qname(col):
    return '"' + col.replace('"', '""') + '"'


def cnt_expr(table):
    cols = table_columns(table)
    if 'CNT' in cols:
        return 'CNT'
    # T22처럼 성별/연령별 count 컬럼으로 나뉜 경우를 위한 보수적 처리
    cnt_cols = [c for c in cols if c.endswith('_CNT')]
    if cnt_cols:
        return ' + '.join(qname(c) for c in cnt_cols)
    return 'NULL'


def date_expr(table):
    cols = table_columns(table)
    if 'ETL_YMD' in cols:
        return 'ETL_YMD'
    if 'ETL_YM' in cols:
        return "TRY_STRPTIME(CAST(ETL_YM AS VARCHAR), '%Y%m')"
    return None


def union_query(parts):
    return '\nUNION ALL\n'.join(parts)

In [ ]:
# 전체 t 데이터 메타데이터: 파일 존재, 행 수, 컬럼 수, 기간 범위
metadata_rows = []
for table, info in T_DATA_FILES.items():
    path = DATA_DIR / info['file']
    if not path.exists():
        metadata_rows.append({'table': table, **info, 'exists': False})
        continue

    d = date_expr(table)
    if d is None:
        row = con.execute(f"SELECT COUNT(*) AS row_count FROM {scan_sql(table)}").fetchone()
        row_count, min_date, max_date = row[0], None, None
    else:
        row_count, min_date, max_date = con.execute(f'''
            SELECT COUNT(*) AS row_count, MIN({d}) AS min_date, MAX({d}) AS max_date
            FROM {scan_sql(table)}
        ''').fetchone()

    metadata_rows.append({
        'table': table,
        **info,
        'exists': True,
        'row_count': row_count,
        'column_count': len(table_columns(table)),
        'min_date': min_date,
        'max_date': max_date,
    })

all_t_metadata = pd.DataFrame(metadata_rows)
all_t_metadata

In [ ]:
# 전체 t 데이터 스키마 목록
schema_rows = []
for table in T_DATA_FILES:
    for col in table_columns(table):
        schema_rows.append({'table': table, 'column': col})

all_t_schema = pd.DataFrame(schema_rows)
all_t_schema

In [ ]:
# 전체 t 데이터 월별 추세: 대용량 스캔이므로 필요할 때만 실행
if RUN_FULL_T_EDA:
    monthly_parts = []
    for table in T_DATA_FILES:
        d = date_expr(table)
        if d is None:
            continue
        monthly_parts.append(f'''
            SELECT
                '{table}' AS table_name,
                STRFTIME(CAST(date_trunc('month', {d}) AS DATE), '%Y-%m') AS ym,
                SUM({cnt_expr(table)}) AS cnt_sum
            FROM {scan_sql(table)}
            GROUP BY 1, 2
        ''')
    all_t_monthly = con.execute(union_query(monthly_parts)).fetchdf()
else:
    all_t_monthly = pd.DataFrame({'안내': ['전체 t 월별 추세는 대용량 스캔이라 기본 실행을 건너뜁니다. RUN_FULL_T_EDA=True로 바꾸면 실행됩니다.']})
all_t_monthly

In [ ]:
# 전체 t 데이터 목적/교통수단/성별연령 코드 분포
if RUN_FULL_T_EDA:
    code_parts = []
    for table in T_DATA_FILES:
        cols = table_columns(table)
        for col in ['PURPOSE', 'TRANS_GB', 'SEX_CD', 'AGE_GRP']:
            if col not in cols:
                continue
            code_parts.append(f'''
                SELECT
                    '{table}' AS table_name,
                    '{col}' AS column_name,
                    CAST({qname(col)} AS VARCHAR) AS code_value,
                    COUNT(*) AS row_count,
                    SUM({cnt_expr(table)}) AS cnt_sum
                FROM {scan_sql(table)}
                GROUP BY 1, 2, 3
            ''')
    all_t_code_distribution = con.execute(union_query(code_parts)).fetchdf()
else:
    all_t_code_distribution = pd.DataFrame({'안내': ['전체 t 코드 분포는 대용량 스캔이라 기본 실행을 건너뜁니다. RUN_FULL_T_EDA=True로 바꾸면 실행됩니다.']})
all_t_code_distribution

In [ ]:
# 전체 t 데이터 지역별 집계
if RUN_FULL_T_EDA:
    region_columns = ['CTY_NM', 'ADMI_NM', 'D_CTY_NM', 'D_ADMI_NM', 'O_CTY_NM', 'O_ADMI_NM']
    region_parts = []
    for table in T_DATA_FILES:
        cols = table_columns(table)
        for col in region_columns:
            if col not in cols:
                continue
            role = '출발' if col.startswith('O_') else '도착' if col.startswith('D_') else '단일지역'
            level = '행정동' if 'ADMI' in col else '시군구'
            region_parts.append(f'''
                SELECT
                    '{table}' AS table_name,
                    '{role}' AS region_role,
                    '{level}' AS region_level,
                    CAST({qname(col)} AS VARCHAR) AS region_name,
                    SUM({cnt_expr(table)}) AS cnt_sum
                FROM {scan_sql(table)}
                WHERE {qname(col)} IS NOT NULL
                GROUP BY 1, 2, 3, 4
            ''')
    all_t_region = con.execute(union_query(region_parts)).fetchdf().sort_values('cnt_sum', ascending=False)
else:
    all_t_region = pd.DataFrame({'안내': ['전체 t 지역별 집계는 대용량 스캔이라 기본 실행을 건너뜁니다. RUN_FULL_T_EDA=True로 바꾸면 실행됩니다.']})
all_t_region

In [ ]:
# 체류시간 분포: T16/T26/T27 등 DURATION 있는 테이블
if RUN_FULL_T_HEAVY_EDA:
    duration_parts = []
    for table in T_DATA_FILES:
        if 'DURATION' not in table_columns(table):
            continue
        duration_parts.append(f'''
            SELECT
                '{table}' AS table_name,
                DURATION,
                COUNT(*) AS row_count,
                SUM({cnt_expr(table)}) AS cnt_sum
            FROM {scan_sql(table)}
            GROUP BY 1, 2
        ''')
    all_t_duration = con.execute(union_query(duration_parts)).fetchdf()
else:
    all_t_duration = pd.DataFrame({'안내': ['전체 t 체류시간 분포는 고비용 스캔이라 기본 실행을 건너뜁니다. RUN_FULL_T_HEAVY_EDA=True로 바꾸면 실행됩니다.']})
all_t_duration

## EDA 진행 흐름

EDA는 전체 T4~T27 구조를 먼저 보고, 그중 예측용 변수로 연결 가능한 테이블을 좁히는 순서로 진행한다.

1. 전체 t 데이터 목록과 단위 확인
2. 테이블별 행 수, 컬럼 수, 기간 범위 확인
3. 목적, 교통수단, 성별, 연령, 시간대, 지역 단위 확인
4. T13/T24/T25/T26/T27을 예측용 변수 후보로 연결
5. 젠트리피케이션 직접 지표가 없으므로 proxy 변수로 해석

In [ ]:
# 전체 t 데이터에서 어떤 컬럼이 어느 테이블에 있는지 확인
column_presence = (
    all_t_schema.assign(present=1)
    .pivot_table(index='column', columns='table', values='present', fill_value=0, aggfunc='max')
    .reset_index()
)
column_presence

In [ ]:
# 예측 변수 후보 테이블만 따로 확인
final_variable_inventory = all_t_metadata[
    all_t_metadata['decision'].eq('예측 변수 후보')
].copy()
final_variable_inventory

## 예측 변수 후보 선택 기준

| 기준 | 설명 |
|---|---|
| 젠트리피케이션 proxy 가능성 | 외부 수요 증가, 상권 활성화, 체류 특성, 이용자 구조 변화와 연결되는가 |
| 행정동 단위 활용성 | 행정동 단위로 다른 데이터와 결합 가능한가 |
| 기간 연속성 | 2023~2025 기간을 안정적으로 커버하는가 |
| 해석 가능성 | 발표/보고서에서 의미를 설명할 수 있는가 |
| 중복성 | 비슷한 의미의 변수끼리 과도하게 중복되지 않는가 |

이 기준으로 T13, T24, T25, T26, T27을 예측 변수 후보 테이블로 남겼다.

## 예측 변수 후보 테이블 상세 EDA

아기존 EDA에서 실제 변수 후보로 연결한 테이블 기준이다.

- T13: 외부유입, 이동량, 목적, 성별/연령
- T24: 유동인구, 목적별 유동, 경제활동 연령층 proxy
- T25: 유입/유출 구조, 목적, 이동수단
- T26: 도착지 체류시간, 목적, 이동수단
- T27: 출발지/이탈 특성, 목적, 이동수단, 체류시간

In [ ]:
# 예측 변수 후보 테이블 월별 행 수와 CNT 합계
final_monthly_rows = []
for table, path in FINAL_FILES.items():
    final_monthly_rows.append(con.execute(f'''
        SELECT
            '{table}' AS table_name,
            STRFTIME(ETL_YMD, '%Y-%m') AS ym,
            COUNT(*) AS row_count,
            SUM(CNT) AS total_cnt
        FROM read_parquet('{p(path)}')
        GROUP BY 1, 2
        ORDER BY 1, 2
    ''').fetchdf())
final_monthly_summary = pd.concat(final_monthly_rows, ignore_index=True)
final_monthly_summary

In [ ]:
# 예측 변수 후보 테이블 목적 분포
purpose_summary = []
for table, path in FINAL_FILES.items():
    purpose_summary.append(con.execute(f'''
        SELECT
            '{table}' AS table_name,
            CAST(PURPOSE AS VARCHAR) AS move_purpose,
            COUNT(*) AS row_count,
            SUM(CNT) AS move_cnt
        FROM read_parquet('{p(path)}')
        GROUP BY 1, 2
        ORDER BY 1, 2
    ''').fetchdf())
purpose_summary = pd.concat(purpose_summary, ignore_index=True)
purpose_summary

In [ ]:
# 예측 변수 후보 테이블 성별/연령 분포: SEX_CD = W는 미확인으로 둔다
sex_age_summary = []
for table, path in FINAL_FILES.items():
    sex_age_summary.append(con.execute(f'''
        SELECT
            '{table}' AS table_name,
            CASE
                WHEN SEX_CD = 'M' THEN 'M'
                WHEN SEX_CD = 'F' THEN 'F'
                WHEN SEX_CD = 'W' THEN 'UNKNOWN_W'
                ELSE 'UNKNOWN'
            END AS sex_group,
            AGE_GRP AS age_group,
            COUNT(*) AS row_count,
            SUM(CNT) AS move_cnt
        FROM read_parquet('{p(path)}')
        WHERE SEX_CD IS NOT NULL AND AGE_GRP IS NOT NULL
        GROUP BY 1, 2, 3
        ORDER BY 1, 2, 3
    ''').fetchdf())
sex_age_summary = pd.concat(sex_age_summary, ignore_index=True)
sex_age_summary

In [ ]:
# 예측 변수 후보 T26/T27 체류시간 + 이동수단 요약
stay_transport_summary = []
for table in ['T26', 'T27']:
    path = FINAL_FILES[table]
    stay_transport_summary.append(con.execute(f'''
        SELECT
            '{table}' AS table_name,
            TRANS_GB AS transport_type,
            PURPOSE AS move_purpose,
            COUNT(*) AS row_count,
            SUM(CNT) AS move_cnt,
            AVG(DURATION) AS stay_time_avg,
            MEDIAN(DURATION) AS stay_time_median
        FROM read_parquet('{p(path)}')
        GROUP BY 1, 2, 3
        ORDER BY 1, 2, 3
    ''').fetchdf())
stay_transport_summary = pd.concat(stay_transport_summary, ignore_index=True)
stay_transport_summary

## 예측 변수 후보 테이블 역할

- T13: 행정동 OD 기준 외부유입, 이동량, 목적, 성별/연령 구조
- T24: 행정동 단위 유동인구, 목적별 유동, 경제활동 연령층 proxy
- T25: 유입·유출 패턴 중심
- T26: 체류시간 중심
- T27: 이동수단 + 목적 + 체류 특성

## 변수 후보 연결표

| 원본 컬럼 | 변수 후보명 | 의미 | 사용 방향 |
|---|---|---|---|
| CNT | move_cnt | 이동량/체류량 규모 | 상권 유입·활동량 |
| DURATION | stay_time | 평균 체류시간 | 체류형 상권 여부 |
| PURPOSE | move_purpose | 이동 목적 | 생활권/업무권 구분 |
| TRANS_GB | transport_type | 이동수단 | 접근성/교통 특성 |
| SEX_CD | sex_group | 성별 구성 | 이용자 특성 |
| AGE_GRP | age_group | 연령대 구성 | 소비층 특성 |

## 예측용 변수 해석 및 선택 기준

현재 통신 데이터에는 임대료, 지가, 공실률, 폐업률처럼 젠트리피케이션을 직접 측정하는 변수가 없다. 따라서 아래 변수들은 젠트리피케이션 자체가 아니라 외부 수요 증가, 상권 활성화, 체류 특성, 거주/이용자 구조 변화의 proxy로 사용한다.

| 후보 변수 | 원천 테이블 | 의미 | 사용 판단 |
|---|---|---|---|
| external_inflow | T13/T25 | 성남시 외부에서 해당 지역으로 들어오는 이동량 | 핵심 후보 |
| move_cnt | T13/T25/T26/T27 | 이동량 또는 체류량 규모 | 핵심 후보 |
| stay_time | T26/T27 | 평균 체류시간 | 핵심 후보 |
| move_purpose | T13/T25/T26/T27 | 이동 목적 | 해석 보조 / 파생변수 후보 |
| transport_type | T25/T26/T27 | 이동수단 | 접근성 proxy 후보 |
| sex_group | T13/T25/T26/T27 | 성별 구성 | 이용자 특성 보조 |
| age_group | T13/T25/T26/T27 | 연령대 구성 | 소비층/생활권 특성 보조 |
| floating_pop | T24 | 행정동 단위 목적별 유동인구 | 보조 후보 |
| working_age_proxy | T13/T24 | 경제활동 연령층 유입/유동 proxy | 보조 후보 |